# CAB ratio from Pythia/FastJet b and light-quark jets

This notebook generates Pythia pp events in memory, tags reconstructed jets by truth flavor, runs the CAB Lund-prong analysis, and compares `CAB(b jets) / CAB(light-quark jets)`.

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def find_repo_dir(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cab_eec").is_dir():
            return path
    raise RuntimeError("Run this notebook from the CAB repository or a subdirectory.")


REPO_DIR = find_repo_dir(Path.cwd())
cab_src = REPO_DIR / "src"
if str(cab_src) not in sys.path:
    sys.path.insert(0, str(cab_src))

utils_src = REPO_DIR.parent / "heppyyier-utils" / "src"
if utils_src.is_dir() and str(utils_src) not in sys.path:
    sys.path.insert(0, str(utils_src))

from heppyyier_utils.cache import ArtifactCache, safe_token
from heppyyier_utils.pythia import PythiaConfig, create_pythia
from heppyyier_utils.pythia.flavor import (
    FlavorTag,
    append_ghosts,
    extract_hard_partons,
    make_heavy_hadron_ghosts,
    require_hadronization_for_ghost_tagging,
    tag_jet_by_hard_parton,
    tag_jet_by_heavy_hadron_ghosts,
)

from cab_eec.eec import EECAccumulator, eec_edges
from cab_eec.event_io import Event
from cab_eec.fastjet_backend import FastJetBackend, JetSplitter, select_lund_splitting
from cab_eec.io_tables import eec_rows

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})

In [ ]:
N_EVENTS = 50000
ECM_GEV = 13_000.0
PTHAT_MIN = 70.0
RANDOM_SEED = 12345
HADRONIZATION = True
PRINT_PYTHIA_STAT = True

JET_R = 0.4
JET_PT_MIN = 95.0
JET_PT_MAX = 125.0
ETA_MAX = 2.0
ETA_MARGIN = 0.05
PT_PARTICLE_MIN = 0.1

FLAVOR_TAG_MODE = "separate_processes"  # "hard_parton", "hadron_ghost", or "separate_processes"
BEAUTY_PROCESS = "hard_qcd_beauty"
LIGHT_PROCESS = "hard_qcd_uds"
FLAVOR_MATCH_RADIUS = 0.3
LIGHT_ABS_IDS = (1, 2, 3)
GLUON_IS_LIGHT = False
CHARM_IS_LIGHT = False
HEAVY_GHOST_SCALE = 1.0e-20
HEAVY_GHOST_USER_INDEX_OFFSET = 10_000_000

RATIO_NUMERATOR = "b"
RATIO_DENOMINATOR = "light"
SAMPLE_LABELS = {
    "b": "b jets",
    "light": "light-quark jets (uds)",
}

SELECTIONS = [
    {"name": "maxkt", "mode": "max_kt"},
    {"name": "sd_z0p1", "mode": "soft_drop", "z_cut": 0.1},
    {"name": "sd_z0p2", "mode": "soft_drop", "z_cut": 0.2},
]
SELECTION_NAMES = [selection["name"] for selection in SELECTIONS]

NORMALIZATIONS = ["radiator_pt2"]

LNDR_MIN = -5.0
LNDR_MAX = 1.5
N_EEC_BINS = 60
RATIO_YLIM = (0.0, 4.0)

USE_DISK_CACHE = True
RECOMPUTE_GENERATION = False
RECOMPUTE_EEC = False

SAVE_FIGURES = False
OUTPUT_DIR = REPO_DIR / "outputs" / "pythia_fastjet_flavor_ratio"
CACHE_DIR = OUTPUT_DIR / "cache"

Flavor-tag modes:

- `hard_parton`: one inclusive hard-QCD sample; jets are matched to outgoing hard partons with configurable `FLAVOR_MATCH_RADIUS`.
- `hadron_ghost`: one inclusive hard-QCD sample; b-jets are tagged by B-hadron ghosts, and light jets are hard-parton-matched uds jets without a B ghost. This requires `HADRONIZATION = True`.
- `separate_processes`: b-jets come from `BEAUTY_PROCESS`, light jets from `LIGHT_PROCESS`, and both are still filtered by hard-parton flavor matches.

Generated splitting records and EEC tables are cached separately under `outputs/pythia_fastjet_flavor_ratio/cache/`; plotting changes do not force event regeneration.

In [ ]:
CACHE_SCHEMA_VERSION = 1


CACHE = ArtifactCache(CACHE_DIR, schema_version=CACHE_SCHEMA_VERSION, base_dir=REPO_DIR)


def compact_generation_stem(config: dict) -> str:
    selections = "_".join(selection["name"] for selection in config["selections"])
    processes = "_".join(item["process"] for item in config.get("process_plan", [])) or "inclusive"
    return safe_token(
        f"{config['flavor_tag_mode']}_{processes}_nev{config['n_events']}_pthat{config['pthat_min']}"
        f"_seed{config['seed']}_R{config['jet_R']}_pt{config['jet_pt_min']}_{config['jet_pt_max']}"
        f"_dr{config['flavor_match_radius']}_{selections}"
    )


def flavor_process_plan() -> list[dict]:
    if FLAVOR_TAG_MODE == "separate_processes":
        return [
            {"process": BEAUTY_PROCESS, "target_sample": "b", "seed_offset": 0},
            {"process": LIGHT_PROCESS, "target_sample": "light", "seed_offset": 1000},
        ]
    return [{"process": "hard_qcd", "target_sample": None, "seed_offset": 0}]


def compact_eec_stem(config: dict) -> str:
    normalizations = "_".join(config["normalizations"])
    return safe_token(
        f"records{config['records_key']}_lndr{config['lndr_min']}_{config['lndr_max']}"
        f"_nb{config['n_bins']}_{normalizations}"
    )


def generation_config() -> dict:
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "generator": "pythia8_pp_flavor_ratio",
        "flavor_tag_mode": FLAVOR_TAG_MODE,
        "process_plan": flavor_process_plan(),
        "n_events": int(N_EVENTS),
        "ecm_gev": float(ECM_GEV),
        "pthat_min": float(PTHAT_MIN),
        "seed": int(RANDOM_SEED),
        "hadronization": bool(HADRONIZATION),
        "jet_R": float(JET_R),
        "jet_pt_min": float(JET_PT_MIN),
        "jet_pt_max": None if JET_PT_MAX is None else float(JET_PT_MAX),
        "eta_max": float(ETA_MAX),
        "eta_margin": float(ETA_MARGIN),
        "pt_particle_min": float(PT_PARTICLE_MIN),
        "flavor_match_radius": float(FLAVOR_MATCH_RADIUS),
        "light_abs_ids": list(LIGHT_ABS_IDS),
        "gluon_is_light": bool(GLUON_IS_LIGHT),
        "charm_is_light": bool(CHARM_IS_LIGHT),
        "heavy_ghost_scale": float(HEAVY_GHOST_SCALE),
        "heavy_ghost_user_index_offset": int(HEAVY_GHOST_USER_INDEX_OFFSET),
        "selections": SELECTIONS,
    }


def generation_cache_path(config: dict | None = None) -> Path:
    config = generation_config() if config is None else config
    return CACHE.path("flavor_splitting_records", compact_generation_stem(config), config)

In [ ]:
def pythia_to_cab_event(pythia, event_id: int) -> Event:
    px: list[float] = []
    py: list[float] = []
    pz: list[float] = []
    energy: list[float] = []
    event = pythia.event
    for index in range(event.size()):
        particle = event[index]
        is_final = bool(particle.isFinal())
        is_visible = bool(particle.isVisible()) if hasattr(particle, "isVisible") else True
        is_parton = bool(particle.isParton()) if hasattr(particle, "isParton") else False
        if is_final and (is_visible or is_parton):
            px.append(float(particle.px()))
            py.append(float(particle.py()))
            pz.append(float(particle.pz()))
            energy.append(float(particle.e()))
    return Event(
        event_id=event_id,
        px=np.asarray(px, dtype=float),
        py=np.asarray(py, dtype=float),
        pz=np.asarray(pz, dtype=float),
        energy=np.asarray(energy, dtype=float),
        weight=1.0,
    )


def create_configured_pythia(process: str, seed_offset: int = 0):
    config = PythiaConfig(
        ecm=ECM_GEV,
        process=process,
        pthat_min=PTHAT_MIN,
        seed=int(RANDOM_SEED) + int(seed_offset),
        hadronization=bool(HADRONIZATION),
    )
    return create_pythia(config, load=True)


def make_splitter(backend: FastJetBackend) -> JetSplitter:
    jet_cfg = {
        "R": JET_R,
        "pt_min": JET_PT_MIN,
        "pt_max": JET_PT_MAX,
        "eta_max": ETA_MAX,
        "eta_margin": ETA_MARGIN,
        "pt_particle_min": PT_PARTICLE_MIN,
    }
    return JetSplitter(backend, jet_cfg, SELECTIONS, "candidate")


def cluster_jets(event: Event, pythia_event, backend: FastJetBackend, splitter: JetSplitter, *, with_ghosts: bool):
    particles = backend.particles_from_event(event)
    ghost_labels = {}
    if with_ghosts:
        ghosts, ghost_labels = make_heavy_hadron_ghosts(
            pythia_event,
            backend.fastjet,
            ghost_scale=HEAVY_GHOST_SCALE,
            user_index_offset=HEAVY_GHOST_USER_INDEX_OFFSET,
        )
        append_ghosts(particles, ghosts)
    if particles.size() == 0:
        return [], ghost_labels, None
    cluster_sequence = backend.fastjet.ClusterSequence(particles, splitter.jet_def)
    jets = backend.fastjet.sorted_by_pt(cluster_sequence.inclusive_jets())
    return jets, ghost_labels, cluster_sequence


def flavor_tag_for_jet(jet, pythia_event, hard_partons, ghost_labels):
    if FLAVOR_TAG_MODE == "hadron_ghost":
        ghost_tag = tag_jet_by_heavy_hadron_ghosts(
            jet,
            ghost_labels,
            user_index_offset=HEAVY_GHOST_USER_INDEX_OFFSET,
        )
        if ghost_tag.flavor == "b":
            return ghost_tag
        hard_tag = tag_jet_by_hard_parton(
            jet,
            hard_partons,
            match_radius=FLAVOR_MATCH_RADIUS,
            light_abs_ids=LIGHT_ABS_IDS,
            gluon_is_light=GLUON_IS_LIGHT,
            charm_is_light=CHARM_IS_LIGHT,
        )
        if hard_tag.flavor == "light":
            return hard_tag
        return FlavorTag("unmatched", source="hadron_ghost")
    return tag_jet_by_hard_parton(
        jet,
        hard_partons,
        match_radius=FLAVOR_MATCH_RADIUS,
        light_abs_ids=LIGHT_ABS_IDS,
        gluon_is_light=GLUON_IS_LIGHT,
        charm_is_light=CHARM_IS_LIGHT,
    )


def records_from_jet(splitter: JetSplitter, event: Event, jet, jet_index: int, sample_name: str) -> list:
    records = []
    sequence = list(splitter.lund_gen.result(jet))
    for selection in SELECTIONS:
        split = select_lund_splitting(sequence, selection)
        if split is None:
            continue
        record = splitter._record_from_split(event, jet, jet_index, selection["name"], split)
        if record is not None:
            records.append(replace(record, sample=sample_name))
    return records


def accepted_sample_from_tag(tag, *, target_sample: str | None = None) -> str | None:
    if tag.flavor not in {"b", "light"}:
        return None
    if target_sample is not None and tag.flavor != target_sample:
        return None
    return tag.flavor

In [ ]:
def generate_records_for_process(process: str, *, target_sample: str | None, seed_offset: int, backend, splitter):
    pythia = create_configured_pythia(process, seed_offset=seed_offset)
    records_by_sample = {"b": [], "light": []}
    stats = {
        "process": process,
        "target_sample": target_sample or "both",
        "generated_events": 0,
        "visible_events": 0,
        "accepted_jets": 0,
        "b_jets": 0,
        "light_jets": 0,
        "unmatched_jets": 0,
        "gluon_jets": 0,
        "charm_jets": 0,
        "other_jets": 0,
    }
    use_ghosts = FLAVOR_TAG_MODE == "hadron_ghost"

    for i_event in tqdm(range(int(N_EVENTS)), desc=f"Pythia {process}", unit="event"):
        if not pythia.next():
            continue
        stats["generated_events"] += 1
        event = pythia_to_cab_event(pythia, i_event + 1_000_000 * int(seed_offset))
        if len(event.px) == 0:
            continue
        stats["visible_events"] += 1

        hard_partons = extract_hard_partons(pythia.event)
        jets, ghost_labels, cluster_sequence = cluster_jets(event, pythia.event, backend, splitter, with_ghosts=use_ghosts)
        if cluster_sequence is None:
            continue

        accepted_jet_index = 0
        for jet in jets:
            if not splitter._accept_jet(jet):
                continue
            stats["accepted_jets"] += 1
            tag = flavor_tag_for_jet(jet, pythia.event, hard_partons, ghost_labels)
            stats[f"{tag.flavor}_jets"] = stats.get(f"{tag.flavor}_jets", 0) + 1
            sample_name = accepted_sample_from_tag(tag, target_sample=target_sample)
            if sample_name is not None:
                records_by_sample[sample_name].extend(records_from_jet(splitter, event, jet, accepted_jet_index, sample_name))
            accepted_jet_index += 1

    if PRINT_PYTHIA_STAT:
        pythia.stat()
    return records_by_sample, stats


def merge_records(target: dict[str, list], source: dict[str, list]) -> None:
    for sample_name, records in source.items():
        target.setdefault(sample_name, []).extend(records)


def generate_splitting_records() -> tuple[dict[str, list], dict]:
    if FLAVOR_TAG_MODE not in {"hard_parton", "hadron_ghost", "separate_processes"}:
        raise ValueError(f"unknown FLAVOR_TAG_MODE: {FLAVOR_TAG_MODE!r}")
    if FLAVOR_TAG_MODE == "hadron_ghost":
        require_hadronization_for_ghost_tagging(HADRONIZATION)

    backend = FastJetBackend.load()
    splitter = make_splitter(backend)
    records_by_sample = {"b": [], "light": []}
    stats_by_process = []

    process_plan = flavor_process_plan()

    for item in process_plan:
        records, stats = generate_records_for_process(
            item["process"],
            target_sample=item["target_sample"],
            seed_offset=int(item["seed_offset"]),
            backend=backend,
            splitter=splitter,
        )
        merge_records(records_by_sample, records)
        stats_by_process.append(stats)

    run_info = {
        "attempted_events_per_process": int(N_EVENTS),
        "flavor_tag_mode": FLAVOR_TAG_MODE,
        "process_plan": process_plan,
        "flavor_match_radius": float(FLAVOR_MATCH_RADIUS),
        "hadronization": bool(HADRONIZATION),
        "backend_versions": backend.versions(),
        "stats_by_process": stats_by_process,
    }
    return records_by_sample, run_info


def load_or_generate_splitting_records() -> tuple[dict[str, list], dict]:
    config = generation_config()
    config_key = CACHE.digest(config)
    path = generation_cache_path(config)
    if USE_DISK_CACHE and path.exists() and not RECOMPUTE_GENERATION:
        payload = CACHE.read_pickle(path)
        run_info = dict(payload.get("run_info", {}))
        run_info.update({"cache_status": "hit", "generation_config_hash": config_key, "generation_cache_path": CACHE.label(path)})
        print(f"[cache hit] flavor splitting records: {CACHE.label(path)}")
        return payload["records_by_sample"], run_info

    records, run_info = generate_splitting_records()
    run_info.update({"cache_status": "miss" if USE_DISK_CACHE else "disabled", "generation_config_hash": config_key, "generation_cache_path": CACHE.label(path)})
    if USE_DISK_CACHE:
        payload = {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "generation_config": config,
            "generation_config_hash": config_key,
            "run_info": run_info,
            "records_by_sample": records,
        }
        CACHE.write_pickle(path, payload, sidecar_exclude={"records_by_sample"})
        print(f"[cache save] flavor splitting records: {CACHE.label(path)}")
    return records, run_info


records_by_sample, run_info = load_or_generate_splitting_records()
run_info

In [ ]:
def mean_or_nan(values: list[float]) -> float:
    return float(np.mean(values)) if values else float("nan")


def radiator_pt(record) -> float:
    return float(record.parent_pt) if record.parent_pt is not None else float(record.pt_a + record.pt_b)


def summarize_records(records_by_sample: dict[str, list]) -> list[dict[str, float | int | str]]:
    rows = []
    for sample_name, records in records_by_sample.items():
        for selection_name in SELECTION_NAMES:
            selected = [record for record in records if record.selection == selection_name]
            rows.append(
                {
                    "sample": sample_name,
                    "label": SAMPLE_LABELS.get(sample_name, sample_name),
                    "selection": selection_name,
                    "n_splittings": len(selected),
                    "n_jets": len({(record.event_id, record.jet_index) for record in selected}),
                    "mean_jet_pt": mean_or_nan([record.jet_pt for record in selected]),
                    "mean_radiator_pt": mean_or_nan([radiator_pt(record) for record in selected]),
                    "mean_Rg": mean_or_nan([record.delta_rg for record in selected]),
                    "mean_z": mean_or_nan([record.z for record in selected]),
                }
            )
    return rows


summary_rows = summarize_records(records_by_sample)
if pd is not None:
    display(pd.DataFrame(summary_rows))
    display(pd.DataFrame(run_info.get("stats_by_process", [])))
else:
    display(summary_rows)
    display(run_info.get("stats_by_process", []))

Accumulate AA, BB, AB, and CAB EECs using the CAB package implementation. EEC tables are cached separately from generated records.

In [ ]:
def eec_analysis_config() -> dict:
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "records_key": run_info.get("generation_config_hash", CACHE.digest(generation_config())),
        "lndr_min": float(LNDR_MIN),
        "lndr_max": float(LNDR_MAX),
        "n_bins": int(N_EEC_BINS),
        "normalizations": list(NORMALIZATIONS),
        "selection_names": list(SELECTION_NAMES),
    }


def eec_cache_path(config: dict | None = None) -> Path:
    config = eec_analysis_config() if config is None else config
    return CACHE.path("flavor_eec_table", compact_eec_stem(config), config)


def compute_eec_table(records_by_sample: dict[str, list]) -> list[dict]:
    edges = eec_edges(LNDR_MIN, LNDR_MAX, N_EEC_BINS)
    rows: list[dict] = []
    for sample_name, records in records_by_sample.items():
        for normalization in NORMALIZATIONS:
            accumulators = {selection_name: EECAccumulator(edges) for selection_name in SELECTION_NAMES}
            for record in tqdm(records, desc=f"{sample_name} EEC {normalization}", unit="split", leave=False):
                accumulator = accumulators.get(record.selection)
                if accumulator is not None:
                    accumulator.fill_record(record, normalization)
            for selection_name, accumulator in accumulators.items():
                rows.extend(eec_rows(sample_name, selection_name, normalization, accumulator.result()))
    return rows


def load_or_compute_eec_table(records_by_sample: dict[str, list]) -> list[dict]:
    config = eec_analysis_config()
    config_key = CACHE.digest(config)
    path = eec_cache_path(config)
    if USE_DISK_CACHE and path.exists() and not RECOMPUTE_EEC:
        payload = CACHE.read_pickle(path)
        print(f"[cache hit] flavor EEC table: {CACHE.label(path)}")
        return payload["eec_table"]
    rows = compute_eec_table(records_by_sample)
    if USE_DISK_CACHE:
        payload = {
            "cache_schema_version": CACHE_SCHEMA_VERSION,
            "eec_config": config,
            "eec_config_hash": config_key,
            "n_rows": len(rows),
            "eec_table": rows,
        }
        CACHE.write_pickle(path, payload, sidecar_exclude={"eec_table"})
        print(f"[cache save] flavor EEC table: {CACHE.label(path)}")
    return rows


eec_table = load_or_compute_eec_table(records_by_sample)
if pd is not None:
    display(pd.DataFrame(eec_table).head())
else:
    display(eec_table[:3])

In [ ]:
def rows_for(sample: str, selection: str, normalization: str) -> list[dict]:
    return sorted(
        [
            row
            for row in eec_table
            if row["sample"] == sample and row["selection"] == selection and row["normalization"] == normalization
        ],
        key=lambda row: row["bin_center_lndr"],
    )


def compatible_bins(first: list[dict], second: list[dict]) -> bool:
    if len(first) != len(second):
        return False
    return all(
        a["bin_lo_lndr"] == b["bin_lo_lndr"] and a["bin_hi_lndr"] == b["bin_hi_lndr"]
        for a, b in zip(first, second)
    )


def ratio_with_uncertainty(numerator, numerator_err, denominator, denominator_err):
    n = np.asarray(numerator, dtype=float)
    sn = np.asarray(numerator_err, dtype=float)
    d = np.asarray(denominator, dtype=float)
    sd = np.asarray(denominator_err, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(d != 0, n / d, np.nan)
        rel_n = np.where(n != 0, sn / n, 0.0)
        rel_d = np.where(d != 0, sd / d, 0.0)
        ratio_err = np.abs(ratio) * np.sqrt(rel_n**2 + rel_d**2)
    ratio_err = np.where(np.isfinite(ratio), ratio_err, np.nan)
    return ratio, ratio_err


def add_linear_rl_axis(ax):
    lo, hi = ax.get_xlim()
    ticks = [tick for tick in (0.01, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4) if lo <= np.log(tick) <= hi]
    if not ticks:
        return
    secax = ax.secondary_xaxis(
        "top",
        functions=(lambda value: np.exp(value), lambda value: np.log(np.maximum(value, 1.0e-12))),
    )
    secax.set_xticks(ticks)
    secax.set_xticklabels([f"{tick:g}" for tick in ticks])
    secax.set_xlabel("R_L")


def maybe_save(fig, stem: str) -> None:
    if not SAVE_FIGURES:
        return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"{stem}.png", bbox_inches="tight")


def finish_lndr_axis(ax):
    ax.set_xlim(LNDR_MIN, LNDR_MAX)
    ax.set_xlabel("ln(delta R)")
    ax.grid(True, alpha=0.25)
    add_linear_rl_axis(ax)


def plot_cab_spectra(selection: str, normalization: str):
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    for sample_name in ("b", "light"):
        group = rows_for(sample_name, selection, normalization)
        if not group:
            continue
        x = np.asarray([row["bin_center_lndr"] for row in group], dtype=float)
        y = np.asarray([row["cab_eec"] for row in group], dtype=float)
        err = np.asarray([row["cab_eec_err"] for row in group], dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if not np.any(mask):
            continue
        n_jets = int(group[0]["n_jets"])
        ax.step(x[mask], y[mask], where="mid", label=f"{SAMPLE_LABELS[sample_name]}, n={n_jets}")
        band = mask & np.isfinite(err)
        ax.fill_between(x[band], y[band] - err[band], y[band] + err[band], step="mid", alpha=0.14)
    finish_lndr_axis(ax)
    ax.set_ylabel("CAB_EEC = AB / sqrt(AA * BB)")
    ax.set_title(f"Pythia pp CAB flavor spectra: {selection}, {normalization}, {FLAVOR_TAG_MODE}")
    ax.legend(fontsize=8)
    maybe_save(fig, f"cab_pythia_flavor__{FLAVOR_TAG_MODE}__{selection}__{normalization}")
    return fig


def plot_cab_ratio(selection: str, normalization: str):
    numerator_rows = rows_for(RATIO_NUMERATOR, selection, normalization)
    denominator_rows = rows_for(RATIO_DENOMINATOR, selection, normalization)
    if not numerator_rows or not denominator_rows:
        print(f"skip {selection}, {normalization}: missing numerator or denominator rows")
        return None
    if not compatible_bins(numerator_rows, denominator_rows):
        raise ValueError(f"incompatible bins for {selection}, {normalization}")
    x = np.asarray([row["bin_center_lndr"] for row in numerator_rows], dtype=float)
    ratio, ratio_err = ratio_with_uncertainty(
        [row["cab_eec"] for row in numerator_rows],
        [row["cab_eec_err"] for row in numerator_rows],
        [row["cab_eec"] for row in denominator_rows],
        [row["cab_eec_err"] for row in denominator_rows],
    )
    mask = np.isfinite(x) & np.isfinite(ratio)
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    ax.axhline(1.0, color="0.35", lw=1.0, ls="--")
    ax.step(x[mask], ratio[mask], where="mid", label=f"CAB({RATIO_NUMERATOR}) / CAB({RATIO_DENOMINATOR})")
    band = mask & np.isfinite(ratio_err)
    ax.fill_between(x[band], ratio[band] - ratio_err[band], ratio[band] + ratio_err[band], step="mid", alpha=0.18)
    finish_lndr_axis(ax)
    ax.set_ylim(*RATIO_YLIM)
    ax.set_ylabel("CAB ratio")
    ax.set_title(f"Pythia pp CAB b/light ratio: {selection}, {normalization}, {FLAVOR_TAG_MODE}")
    ax.legend(fontsize=8)
    maybe_save(fig, f"ratio_cab_pythia_flavor__{FLAVOR_TAG_MODE}__{selection}__{normalization}")
    return fig

The ratio below is `CAB(b jets) / CAB(light-quark jets)` with propagated uncertainties. Change `FLAVOR_TAG_MODE`, `FLAVOR_MATCH_RADIUS`, or EEC settings above and rerun the analysis cells to compare alternatives.

In [ ]:
for normalization in NORMALIZATIONS:
    for selection in SELECTION_NAMES:
        spectra_fig = plot_cab_spectra(selection, normalization)
        display(spectra_fig)
        plt.close(spectra_fig)

        ratio_fig = plot_cab_ratio(selection, normalization)
        if ratio_fig is not None:
            display(ratio_fig)
            plt.close(ratio_fig)